# Drex evaluators for LlamaIndex

Replace your LLM judge: Drex returns probabilities, not parsed text.

Install (coming soon to PyPI; from source meanwhile):

```
pip install "git+https://github.com/nace-ai/llama-index-nace#subdirectory=packages/llama-index-evaluation-nace"
```

Set `NACE_API_KEY` in the environment before running.

The default model is `drex-max-v1.1`.

In [1]:
from llama_index.evaluation.nace import (
    NaceCorrectnessEvaluator,
    NaceFaithfulnessEvaluator,
    NaceRelevancyEvaluator,
)

query = "When was the Eiffel Tower completed?"
contexts = ["The Eiffel Tower, in Paris, was completed in March 1889 for the World's Fair."]
good = "The Eiffel Tower was completed in 1889."
bad = "The Eiffel Tower was completed in 1925 and is made of concrete."

## Faithfulness and relevancy

`evaluate` is sync and works inside a notebook's running event loop. `score` is P(yes); `passing` is `score >= threshold` (default 0.5).

In [2]:
faithfulness = NaceFaithfulnessEvaluator()
relevancy = NaceRelevancyEvaluator()
for answer in (good, bad):
    f = faithfulness.evaluate(query=query, response=answer, contexts=contexts)
    r = relevancy.evaluate(query=query, response=answer, contexts=contexts)
    print(answer)
    print("  ", f.passing, f.feedback, "|", r.passing, r.feedback)
print(f.model, f.request_id is not None, f.usage)

The Eiffel Tower was completed in 1889.
   True P(faithful)=0.97 | True P(relevant)=0.91


The Eiffel Tower was completed in 1925 and is made of concrete.
   False P(faithful)=0.01 | False P(relevant)=0.01
drex-max-v1.1 True input_tokens=178 output_tokens=36 image_tokens=None


## Correctness against a reference

A 1 to 5 score (expected level), passing at `score_threshold` (default 4.0). Pass `reference=` to judge against a reference answer.

In [3]:
correctness = NaceCorrectnessEvaluator()
for answer in (good, bad):
    c = correctness.evaluate(query=query, response=answer, reference="It opened in 1889.")
    print(c.score, c.passing, c.feedback)
    print("  level probabilities:", c.probabilities)

4.505571583713923 True score=4.51, P(≥4)=0.96
  level probabilities: {0: 0.00542126593565308, 1: 0.0066444891090360995, 2: 0.023037921418735407, 3: 0.40673404237888555, 4: 0.5581622811576898}


2.197156068384303 False score=2.20, P(≥4)=0.03
  level probabilities: {0: 0.022359186200371743, 1: 0.7955670745336655, 2: 0.1562397141245082, 3: 0.014226534964197336, 4: 0.011607490177257243}


## Evaluating a RAG response

`evaluate_response` reads the text source nodes as contexts, so it plugs into `BatchEvalRunner`. Here we build a `Response` by hand.

In [4]:
from llama_index.core.base.response.schema import Response
from llama_index.core.schema import NodeWithScore, TextNode

response = Response(
    response=good,
    source_nodes=[NodeWithScore(node=TextNode(text=contexts[0]), score=0.9)],
)
result = faithfulness.evaluate_response(query=query, response=response)
print(result.passing, result.feedback)

True P(faithful)=0.97


## Images

When the sources contain an `ImageNode`, the evaluators send the image; the default model, `drex-max-v1.1`, reads it (if `NACE_DEFAULT_DECISION_MODEL` names a text-only model, image requests still go to `drex-max-v1.1`). Images come from `image_path` or a base64 `image`, or a `data:` URL; `http(s)` URLs are refused because Drex never fetches URLs. More than 8 images needs `max_images=N` (keeps the N highest-scored). Here we draw a tiny bar chart. Faithfulness is strict, so a chart-only source scores lower than text: the true claim gets a middling probability and the false one a near-zero one. Compare probabilities, or lower `threshold`, rather than relying on the default cut-off for image-only sources.

In [5]:
import tempfile
from pathlib import Path

from llama_index.core.schema import ImageNode
from PIL import Image, ImageDraw

img = Image.new("RGB", (320, 200), "white")
draw = ImageDraw.Draw(img)
draw.text((90, 8), "Quarterly sales", fill="black")
for i, (h, name) in enumerate([(50, "Q1"), (130, "Q2")]):
    draw.rectangle([60 + i * 120, 170 - h, 140 + i * 120, 170], fill="steelblue")
    draw.text((92 + i * 120, 176), name, fill="black")
path = Path(tempfile.mkdtemp()) / "chart.png"
img.save(path)

for claim in ("Q2 sales are higher than Q1.", "Q1 sales are higher than Q2."):
    chart = Response(
        response=claim,
        source_nodes=[NodeWithScore(node=ImageNode(image_path=str(path)), score=1.0)],
    )
    result = faithfulness.evaluate_response(query="Which quarter sold more?", response=chart)
    print(claim, "|", result.model, result.passing, result.feedback)

Q2 sales are higher than Q1. | drex-max-v1.1 False P(faithful)=0.40


Q1 sales are higher than Q2. | drex-max-v1.1 False P(faithful)=0.02
